# Episode 8: Pipelines & Hierarchies

Some tasks are naturally sequential — like an assembly line where each station does one thing.

**By the end of this episode, you will:**
- Build a document processing pipeline where agents transform data in sequence
- Build a hierarchical research team with a manager delegating to specialists
- Know when to reach for a pipeline vs a hierarchy

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig

## Two structural patterns

So far we've seen agents in groups — round-robin, auto, handoffs. Now we shift from *who speaks next* to *how agents are organized*.

A **sequential pipeline** is a chain where each agent transforms the data and passes it forward. Think document processing: draft, review, polish. A **hierarchy** is a tree where a manager delegates to specialists, then synthesizes their results. Think research: gather data, analyze it, write the report.

Both use `DefaultPattern` with explicit handoffs under the hood. The difference is the **topology** — how the agents are wired together.

---

## Part 1: Sequential Pipeline

A pipeline is a linear chain where each agent transforms the output and passes it to the next:

```
draft_agent  -->  review_agent  -->  polish_agent  -->  DONE
```

Each agent has exactly one handoff: to the next agent in the chain. The last agent terminates. Simple, predictable, easy to debug.

### Step 1: Create pipeline agents

We'll build a three-stage content pipeline. The draft agent writes, the review agent critiques and rewrites, and the polish agent makes it publication-ready. Let's see this:

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig
from autogen.agentchat.group import AgentTarget, OnCondition, StringLLMCondition
from autogen.agentchat.group import TerminateTarget

llm_config = LLMConfig({"model": "gpt-4o-mini"})

polish_agent = ConversableAgent(
    name="polish_agent",
    system_message=(
        "You are an editor who polishes text. Take the reviewed draft and make it publication-ready: "
        "fix grammar, improve flow, ensure consistent tone. Output the final version."
    ),
    description="Final editor. Polishes text for publication.",
    llm_config=llm_config, human_input_mode="NEVER",
)

review_agent = ConversableAgent(
    name="review_agent",
    system_message=(
        "You are a content reviewer. Read the draft and provide specific feedback: "
        "what works, what needs improvement, and concrete suggestions. "
        "Then rewrite the draft incorporating your feedback."
    ),
    description="Reviews and improves drafts.",
    llm_config=llm_config, human_input_mode="NEVER",
)

draft_agent = ConversableAgent(
    name="draft_agent",
    system_message=(
        "You are a content writer. Write a first draft based on the given topic. "
        "Focus on getting the ideas down clearly. Aim for 2-3 paragraphs."
    ),
    description="Writes first drafts.",
    llm_config=llm_config, human_input_mode="NEVER",
)

### Step 2: Wire up the pipeline with handoffs

Here's the key part — each agent's `after_work` points to the next stage. No conditions, no branching, just a straight line.

In [ ]:
# draft -> review (always)
draft_agent.set_after_work(AgentTarget(review_agent))

# review -> polish (always)
review_agent.set_after_work(AgentTarget(polish_agent))

# polish -> terminate
polish_agent.set_after_work(TerminateTarget())

### Step 3: Run the pipeline

Let's feed in a topic and watch the assembly line do its work.

In [ ]:
from autogen.agentchat import initiate_group_chat
from autogen.agentchat.group.patterns import DefaultPattern

user = ConversableAgent(name="user", llm_config=llm_config, human_input_mode="NEVER")

pattern = DefaultPattern(
    initial_agent=draft_agent,
    agents=[draft_agent, review_agent, polish_agent],
)

result, context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Write a short blog post about why pair programming improves code quality.",
    max_rounds=4,
)

### What happened

The document flowed through three stages: **draft_agent** wrote the initial draft, **review_agent** critiqued it and rewrote it, and **polish_agent** made it publication-ready.

Notice each agent only sees the previous agent's output, not the whole conversation. This is by design — it keeps each stage focused on its job. The chain terminated automatically after `polish_agent` via `TerminateTarget()`. No orchestrator needed.

---

## Part 2: Hierarchical (Manager + Specialists)

A hierarchy has a **manager** that delegates to specialists and synthesizes their results:

```
         research_manager
           /          \
    data_agent    analysis_agent
```

Unlike a pipeline, the manager can re-delegate — asking for more data or deeper analysis as the task evolves. This makes hierarchies a good fit when the number and order of subtasks aren't known in advance.

### Step 1: Create the team

The manager uses `OnCondition` handoffs to delegate. Specialists just do their job and report back.

In [ ]:
data_agent = ConversableAgent(
    name="data_agent",
    system_message=(
        "You are a data specialist. When asked, provide relevant statistics, "
        "metrics, and data points. Cite sources when possible. Be concise."
    ),
    description="Provides data, statistics, and metrics.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

analysis_agent = ConversableAgent(
    name="analysis_agent",
    system_message=(
        "You are an analyst. When given data or a topic, provide insightful analysis: "
        "identify trends, draw conclusions, and make recommendations. Be concise."
    ),
    description="Analyzes data and provides insights and recommendations.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

research_manager = ConversableAgent(
    name="research_manager",
    system_message=(
        "You are a research manager. Break down research questions into data-gathering "
        "and analysis tasks. Delegate to your team, then synthesize their findings "
        "into a final report. Keep your synthesis to 2-3 paragraphs."
    ),
    description="Manages research projects. Delegates to data and analysis specialists.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

### Step 2: Specialists report back to manager

After each specialist finishes, control flows back to the manager. This is what creates the hub-and-spoke topology.

In [ ]:
# After specialists finish, route back to the manager
data_agent.set_after_work(AgentTarget(research_manager))
analysis_agent.set_after_work(AgentTarget(research_manager))

### Step 3: Run the research team

Let's give the manager a research question and watch it delegate.

In [ ]:
pattern = DefaultPattern(
    initial_agent=research_manager,
    agents=[research_manager, data_agent, analysis_agent],
)

result, context, last_agent = initiate_group_chat(
    pattern=pattern,
    messages="Research the impact of remote work on developer productivity.",
    max_rounds=6,
)

### What happened

The **research_manager** received the question, broke it down, and delegated data gathering to **data_agent**. After getting the statistics, the manager sent them to **analysis_agent** for interpretation. Finally, the manager synthesized everything into a cohesive report.

The key difference from a pipeline: the manager can **re-delegate**. If the analysis reveals gaps, the manager can send the data agent back for more numbers. This adaptive loop is what makes hierarchies powerful for complex, open-ended tasks.

## Choosing between them

The choice comes down to whether the steps are fixed or adaptive.

| Pattern | Topology | Best For |
|---------|----------|----------|
| **Pipeline** | Linear chain (A -> B -> C) | Data transformation, document processing, approval chains, ETL |
| **Hierarchy** | Tree (Manager -> Specialists) | Reports, complex analysis, tasks requiring delegation and synthesis |

If you can number the steps 1-2-3 before the task starts, use a pipeline. If the manager needs to decide the next step based on what just happened, use a hierarchy.

## Additional Content

### Star pattern

The star is just a hierarchy without middle management. One coordinator sits at the center with flat specialists around it — no intermediate aggregation layer.

```
    specialist_1
        |
coordinator --- specialist_2
        |
    specialist_3
```

Implementation-wise, it's identical to the hierarchy example above — the difference is conceptual. Use a star when one coordinator can handle all the synthesis. Graduate to a full hierarchy when you need intermediate managers (e.g., a VP delegates to team leads who delegate to engineers) because the coordinator would be overwhelmed doing everything itself.

## What's Next

You now have five orchestration patterns in your toolkit: round-robin, auto, handoffs, pipelines, and hierarchies. But how do you know which one to reach for when you're staring at a new problem?

In **Episode 9**, you'll learn **how to choose between all these patterns** — a decision framework for picking the right orchestration approach.

**Bonus:** Try the sequential pipeline and nested chat demos in the [AG2 Playground](https://playground.ag2.ai).